# Diagnostico: por que FedAvg empeora al terminar la exploracion

En el barrido de K, FedAvg mejora mientras explora y se cae cuando ε llega a su piso de 0.05 (episodio 2400 de 4000). En 5 de los 6 K termina peor que cuando actuaba casi al azar, y por debajo de DTE con K = 5 y 10. Este notebook compara cuatro variantes del entrenamiento con K = 10 y la semilla 0, lanzadas con `make run-diag`:

| Variante | Que cambia | Que pone a prueba |
|---|---|---|
| base | nada | que la caida se repite con 2000 episodios, donde ε llega al piso en el episodio 1200 |
| buffer100k | replay buffer de 100 000 transiciones (500 episodios) en vez de 10 000 (50) | que el buffer chico se llena solo con la politica del momento cuando ε baja, y la red olvida lo que valian las demas acciones |
| reward0.01 | los agentes aprenden de la recompensa x 0.01 | que la recompensa sin escalar (17 a 35 por slot) deja los valores Q en miles y la perdida Huber trabaja como L1 |
| ambos | las dos cosas | si los dos efectos se suman |

**Que buscar.** Una variante que (1) no se caiga cuando ε llega al piso, (2) termine por encima de su arranque casi al azar y (3) le gane a Greedy y a DTE con K = 10. Esa seria la configuracion para volver a correr FedAvg, DTE y gossip.

**Referencias.** Greedy y DTE con K = 10 y la semilla 0, del barrido (`runs/k10/`). DTE corrio 4000 episodios y estas variantes 2000, asi que su linea es una referencia, no una comparacion exacta. Greedy no aprende, asi que su valor no depende de los episodios.

**Datos.** En Hypatia, desde la raiz del repo: `make run-diag`. Cuando termine, en la maquina local: `make pull-runs`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # v2v/, para importar analysis.py y plots.py
from analysis import final_values, load_metadata, load_runs, per_episode_means
from plots import LABELS, METHODS, REFERENCE, use_thesis_style

RUNS_ROOT = Path(os.environ.get("V2V_RUNS_ROOT", "../runs"))
FIG_DIR = Path(os.environ.get("V2V_FIG_DIR", "../runs/figures"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_K, DIAG_SEED = 10, 0
VARIANTS = {  # nombre en runs/diag/ -> etiqueta y color
    "base": ("Base", "#2a78d6"),
    "buffer100k": ("Buffer de 100 000", "#8a4fd6"),
    "reward0.01": ("Recompensa x 0.01", "#d6457b"),
    "ambos": ("Las dos", "#16917d"),
}
WINDOW = 50  # episodios de la media movil
FINAL_FRACTION = 0.1  # tramo final y tramo inicial: el 10 % de los episodios
EPS_FLOOR_FRACTION = 0.6  # main.epsilon_schedule: epsilon llega a 0.05 al 60 % del entrenamiento
COLLAPSE_TOLERANCE = 0.05  # "se cae": el final queda mas de un 5 % por debajo de su mejor tramo
use_thesis_style()

## Carga

Una fila por variante: cuantos episodios tiene y cuanto tardo. Las referencias del barrido aparecen abajo; si falta alguna, las secciones que la usan lo dicen.

In [ ]:
frames, meta_rows = [], []
for name in VARIANTS:
    root = RUNS_ROOT / "diag" / name
    if not (root / f"k{DIAG_K}" / f"fedavg_seed{DIAG_SEED}.csv").exists():
        print(f"Falta la variante {name} (se esperaba {root}/k{DIAG_K}/fedavg_seed{DIAG_SEED}.csv).")
        continue
    run = per_episode_means(load_runs(root, methods=["fedavg"]))
    frames.append(run[(run["k"] == DIAG_K) & (run["seed"] == DIAG_SEED)].assign(variant=name))
    meta_rows.append(load_metadata(root, methods=["fedavg"]).assign(variant=name))
if not frames:
    raise FileNotFoundError(f"No hay ninguna variante en {RUNS_ROOT / 'diag'}: corre `make run-diag` y `make pull-runs`.")
diag = pd.concat(frames, ignore_index=True)
variants = [v for v in VARIANTS if v in set(diag["variant"])]
episodes = int(diag["episode"].max()) + 1
eps_floor_episode = EPS_FLOOR_FRACTION * episodes

meta = pd.concat(meta_rows, ignore_index=True).set_index("variant")
coverage = pd.DataFrame(
    {
        "Episodios": diag.groupby("variant")["episode"].count(),
        "Buffer": meta["buffer_capacity"] if "buffer_capacity" in meta else np.nan,
        "Escala de la recompensa": meta["reward_scale"] if "reward_scale" in meta else np.nan,
        "Horas": meta["wall_time_s"] / 3600,
    }
).reindex(variants)
coverage.index = [VARIANTS[v][0] for v in coverage.index]

references = {}
try:
    sweep = per_episode_means(load_runs(RUNS_ROOT, methods=["greedy", "dte"]))
    sweep = sweep[(sweep["k"] == DIAG_K) & (sweep["seed"] == DIAG_SEED)]
    for method, run in sweep.groupby("method"):
        references[method] = final_values(run, FINAL_FRACTION).iloc[0]
except FileNotFoundError:
    pass
missing_refs = [m for m in ("greedy", "dte") if m not in references]
if missing_refs:
    print(f"Sin referencia del barrido para: {', '.join(missing_refs)} (K = {DIAG_K}, semilla {DIAG_SEED}).")
coverage.round(2)

## Curvas de aprendizaje

Recompensa por episodio (media movil de `WINDOW` episodios). La linea vertical marca el episodio en que ε llega a 0.05. En el barrido, la caida empezaba poco antes de ese punto. Lo que hay que ver: cual variante sigue subiendo, o al menos se mantiene, despues de la linea.

In [ ]:
def moving(run: pd.DataFrame, metric: str) -> pd.Series:
    run = run.sort_values("episode")
    return run.set_index("episode")[metric].rolling(WINDOW, min_periods=WINDOW).mean()


fig, ax = plt.subplots(figsize=(7.5, 4.2))
for name in variants:
    label, color = VARIANTS[name]
    ax.plot(moving(diag[diag["variant"] == name], "reward"), color=color, label=label)
for method, ref in references.items():
    style = METHODS[method]
    ax.axhline(ref["reward"], color=style["color"], linestyle="--", linewidth=1.2, label=f"{style['label']} (final del barrido)")
ax.axvline(eps_floor_episode, color=REFERENCE, linestyle=":", linewidth=1)
ax.annotate("ε = 0.05", (eps_floor_episode, 1), xycoords=("data", "axes fraction"), xytext=(4, -12),
            textcoords="offset points", color=REFERENCE, fontsize=8)
ax.set_xlabel("Episodio")
ax.set_title(f"{LABELS['reward']}, K = {DIAG_K}, semilla {DIAG_SEED}")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=7.5, ncol=3)
fig.tight_layout()
fig.savefig(FIG_DIR / "diagnostico_curvas.pdf")
plt.show()

## Que cambia en el comportamiento

Las mismas curvas para tasa CUE, outage y potencia media. En el barrido, la caida de FedAvg venia con menos potencia, menos canales y mucho mas outage: los pares dejaban de transmitir para proteger a los CUEs. Una variante que arregla el problema deberia mantener el outage bajo despues de la linea.

In [ ]:
panels = [("cue_rate_mbps", 1), ("outage_rate", 100), ("mean_power_ratio", 1)]
fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.9))
for ax, (metric, factor) in zip(axes, panels):
    for name in variants:
        label, color = VARIANTS[name]
        ax.plot(moving(diag[diag["variant"] == name], metric) * factor, color=color, label=label)
    for method, ref in references.items():
        ax.axhline(ref[metric] * factor, color=METHODS[method]["color"], linestyle="--", linewidth=1.1)
    ax.axvline(eps_floor_episode, color=REFERENCE, linestyle=":", linewidth=1)
    ax.set_xlabel("Episodio")
    ax.set_title(LABELS[metric])
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "diagnostico_comportamiento.pdf")
plt.show()

## Tabla

- **Casi al azar**: promedio del primer 10 % de los episodios, cuando ε todavia esta entre 1 y ~0.85.
- **Mejor tramo**: el maximo de la media movil de `WINDOW` episodios.
- **Final**: promedio del ultimo 10 %, como en el resto de la tesis.
- **Caida**: cuanto queda el final por debajo del mejor tramo, en %.

In [ ]:
rows = {}
for name in variants:
    run = diag[diag["variant"] == name].sort_values("episode")
    n = max(1, int(np.ceil(FINAL_FRACTION * len(run))))
    start, end = run.head(n), run.tail(n)
    best = moving(run, "reward").max()
    row = {
        "Casi al azar": start["reward"].mean(),
        "Mejor tramo": best,
        "Final": end["reward"].mean(),
        "Caida (%)": 100 * (end["reward"].mean() - best) / abs(best),
    }
    for method, ref in references.items():
        row[f"Final - {METHODS[method]['label']}"] = end["reward"].mean() - ref["reward"]
    row[LABELS["cue_rate_mbps"]] = end["cue_rate_mbps"].mean()
    row[LABELS["outage_rate"]] = 100 * end["outage_rate"].mean()
    row[LABELS["mean_power_ratio"]] = end["mean_power_ratio"].mean()
    row[LABELS["channels_used"]] = end["channels_used"].mean()
    rows[VARIANTS[name][0]] = row
table = pd.DataFrame(rows).T
for method, ref in references.items():
    reference_row = {
        "Final": ref["reward"],
        LABELS["cue_rate_mbps"]: ref["cue_rate_mbps"],
        LABELS["outage_rate"]: 100 * ref["outage_rate"],
        LABELS["mean_power_ratio"]: ref["mean_power_ratio"],
        LABELS["channels_used"]: ref["channels_used"],
    }
    table.loc[f"{METHODS[method]['label']} (referencia)"] = pd.Series(reference_row)
table.round(2)

## Veredicto

Una linea por variante con los tres criterios. La primera que los cumpla todos es la candidata a configuracion de la tesis; si ninguna los cumple, la siguiente hipotesis es la de los agentes sincronizados (todos comparten el mismo modelo y deciden lo mismo a la vez).

In [ ]:
passing = []
for name in variants:
    label = VARIANTS[name][0]
    row = table.loc[label]
    collapses = row["Caida (%)"] < -100 * COLLAPSE_TOLERANCE
    learns = row["Final"] > row["Casi al azar"]
    beats = {m: row["Final"] > references[m]["reward"] for m in references}
    checks = [
        "se cae al terminar la exploracion" if collapses else "no se cae",
        "termina por encima de su arranque" if learns else "termina por debajo de su arranque casi al azar",
    ]
    checks += [("le gana a " if ok else "no le gana a ") + METHODS[m]["label"] for m, ok in beats.items()]
    print(f"{label}: {', '.join(checks)} (caida {row['Caida (%)']:.1f} %).")
    if not collapses and learns and beats and all(beats.values()):
        passing.append(label)
print()
if passing:
    print(f"Cumplen los tres criterios: {', '.join(passing)}.")
else:
    print("Ninguna variante cumple los tres criterios.")